# Tarea 6 · Inspección y limpieza de TotalCharges

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/feat/6-totalcharges/notebooks/trabajo/6_totalcharges.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel. **Responsable de esta tarea:** Julián.

Inspeccionamos el dataset original y generamos `data/processed/telco_step1.csv` para las tareas 7 y 8.

## Cómo ejecutarlo

- **Local:** instalá las dependencias de `requirements.txt`, abrí el notebook desde el repositorio y ejecutá todas las celdas.
- **Colab:** abrí el badge y ejecutá todas las celdas. El notebook lee el CSV original desde GitHub y guarda el resultado en `data/processed/` del entorno de Colab. Podés descargarlo desde el panel Archivos.


## 1. Carga e inspección del dataset original

En Colab leemos el CSV desde GitHub. En local lo buscamos desde el directorio de ejecución y sus padres. Revisamos dimensiones, tipos, estadísticas descriptivas y duplicados para conocer el estado inicial del dataset.


In [ ]:
from pathlib import Path

import pandas as pd

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

GITHUB_REPO = 'julianschmidt1/tpi-analisis-de-datos'
GITHUB_REF = 'main'
RUTA_RAW = Path('data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
URL_RAW = f'https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_REF}/{RUTA_RAW.as_posix()}'

if EN_COLAB:
    RAIZ = Path.cwd()
    fuente = URL_RAW
else:
    actual = Path.cwd().resolve()
    RAIZ = next(
        (ruta for ruta in (actual, *actual.parents) if (ruta / RUTA_RAW).is_file()),
        None,
    )
    if RAIZ is None:
        raise FileNotFoundError(
            f'No se encontró {RUTA_RAW.as_posix()}. Abrí Jupyter dentro del repositorio '
            'y comprobá que esté disponible el dataset original de la tarea 2.'
        )
    fuente = RAIZ / RUTA_RAW

df = pd.read_csv(fuente)
print(f'Entorno: {"Colab" if EN_COLAB else "local"}')
print(f'Fuente: {fuente}')
print(f'Dataset original: {df.shape[0]:,} filas y {df.shape[1]} columnas.')
df.info()
print(df.describe(include="all").T.to_string())
print(f"Filas duplicadas: {df.duplicated().sum()}")
print(f"IDs de cliente repetidos: {df['customerID'].duplicated().sum()}")

El archivo contiene 7.043 registros y 21 variables. La inspección de tipos muestra que Pandas lee `TotalCharges` como texto. Para analizar la facturación acumulada, primero hay que identificar los valores vacíos y convertir la columna a numérica.

## 2. Conversión y diagnóstico de faltantes

Quitamos espacios de los extremos, convertimos las cadenas vacías a faltantes y aplicamos `to_numeric`. La conversión genera un error si encuentra texto inesperado, para que el tratamiento de vacíos no oculte un dato inválido. Luego revisamos la antigüedad y el estado de baja de los clientes afectados.

In [ ]:
total_charges = df["TotalCharges"].str.strip().replace("", pd.NA)
df["TotalCharges"] = pd.to_numeric(total_charges, errors="raise").astype("float64")

faltantes = df["TotalCharges"].isna()
print(f"TotalCharges faltantes: {faltantes.sum()} ({faltantes.mean():.2%})")
print(df.loc[faltantes, ["tenure", "Churn"]].value_counts().rename("clientes"))

## 3. Decisión y exportación

Los 11 faltantes (0,16 % del total) corresponden a clientes con `tenure = 0` y `Churn = No`. Imputamos `0.0` porque usamos cero meses registrados como indicio de que todavía no hay cargos acumulados. Así conservamos a esos clientes en el análisis. El cero es un valor imputado a partir de la antigüedad y su validez depende de esa suposición. Solo aplicamos la regla cuando falta `TotalCharges` y la antigüedad es cero. Antes de exportar, verificamos el tipo y que no queden faltantes.

In [ ]:
df.loc[faltantes & df["tenure"].eq(0), "TotalCharges"] = 0.0

assert df["TotalCharges"].dtype == "float64"
assert df["TotalCharges"].notna().all(), "Quedaron faltantes de TotalCharges por revisar."

ruta_salida = RAIZ / "data" / "processed" / "telco_step1.csv"
ruta_salida.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(ruta_salida, index=False)
print(f"Exportado: {ruta_salida.name} ({len(df)} filas)")